# Partie 1 : ABA generator
Le moteur est dans `aba.py` ; ce notebook démontre chaque capacité demandée.

## 1. Parsing de l'exemple du sujet

In [1]:
from aba import *
fw = ABAFramework.from_text(EXAMPLE)
print(fw.to_text())
print('plat:', fw.is_flat(), '| circulaire:', fw.is_circular(), '| atomique:', fw.is_atomic())

## Literals
L: [a,b,c,q,p,r,s,t]

## Assumptions
A: [a,b,c]

## Contraries
C(a): r
C(b): s
C(c): t

## Rules
[r1]: p <- q,a
[r2]: q <-
[r3]: r <- b,c
[r4]: t <- p,c
[r5]: s <- t

## Preferences
PREF: a > b

plat: True | circulaire: False | atomique: False


## 2. Conversion atomique (dépliage des règles)

In [2]:
at = fw.prepare()
print(at.to_text())
print('atomique:', at.is_atomic())

## Literals
L: [a,b,c,q,p,r,s,t]

## Assumptions
A: [a,b,c]

## Contraries
C(a): r
C(b): s
C(c): t

## Rules
[u1]: p <- a
[u2]: q <-
[u3]: r <- b,c
[u4]: s <- a,c
[u5]: t <- a,c

## Preferences
PREF: a > b

atomique: True


## 3. Arguments (avec arbres de dérivation) et attaques sans préférences

In [3]:
args = fw.arguments()
for a in args: print(a)
print()
print(args[7].tree_str())
print()
for t in fw.attacks(args, use_preferences=False): print(t)

A1: {} ⊢ q
A2: {a} ⊢ a
A3: {b} ⊢ b
A4: {c} ⊢ c
A5: {a} ⊢ p
A6: {b,c} ⊢ r
A7: {a,c} ⊢ s
A8: {a,c} ⊢ t

t   [r4]
  p   [r1]
    q   [r2]
    a   (hypothèse)
  c   (hypothèse)

A6 -> A2  (normal, via a)
A6 -> A5  (normal, via a)
A6 -> A7  (normal, via a)
A6 -> A8  (normal, via a)
A7 -> A3  (normal, via b)
A7 -> A6  (normal, via b)
A8 -> A4  (normal, via c)
A8 -> A6  (normal, via c)
A8 -> A7  (normal, via c)
A8 -> A8  (normal, via c)


## 4. Préférences `a > b` : attaques normales / inversées

In [4]:
for t in fw.attacks(args): print(t)

A2 -> A6  (reverse, via b)
A5 -> A6  (reverse, via b)
A7 -> A3  (normal, via b)
A7 -> A6  (normal, via b)
A7 -> A6  (reverse, via b)
A8 -> A4  (normal, via c)
A8 -> A6  (normal, via c)
A8 -> A6  (reverse, via b)
A8 -> A7  (normal, via c)
A8 -> A8  (normal, via c)


In [5]:
from collections import Counter
print(Counter(t.kind for t in fw.attacks(args)))

Counter({'normal': 6, 'reverse': 4})


## 5. Cas supplémentaires
### 5a. Règles circulaires

In [6]:
circ = ABAFramework.from_text('''
L: [a,b,p,q,r]
A: [a,b]
C(a): q
C(b): r
[r1]: p <- q,a
[r2]: q <- p
[r3]: q <- b
''')
print('cycle:', circ.find_cycle())
nc = circ.to_non_circular(); print('non circulaire après conversion:', not nc.is_circular())
print(nc.to_atomic().to_text())
for a in circ.arguments(): print(a)

cycle: ['p', 'q', 'p']
non circulaire après conversion: True
## Literals
L: [a,b,r,q,p]

## Assumptions
A: [a,b]

## Contraries
C(a): q
C(b): r

## Rules
[u1]: p <- a,b
[u2]: q <- b

A1: {a} ⊢ a
A2: {b} ⊢ b
A3: {b} ⊢ q
A4: {a,b} ⊢ p


### 5b. Hypothèse en tête de règle (conversion en ABA plat)

In [7]:
nf = ABAFramework.from_text('''
L: [a,b,c,x]
A: [a,b]
C(a): c
C(b): x
[r1]: a <- b
[r2]: c <-
''')
print('plat:', nf.is_flat()); print(nf.to_flat().to_text())

plat: False
## Literals
L: [a,b,c,x,a']

## Assumptions
A: [a',b]

## Contraries
C(a'): c
C(b): x

## Rules
[r1]: a <- b
[r2]: c <-
[f_a]: a <- a'



### 5c. Préférences d'équivalence `a,b > c` et chaînes `a > b > c`

In [8]:
p = ABAFramework.from_text('''
L: [a,b,c,x,y,z]
A: [a,b,c]
C(a): x
C(b): y
C(c): z
[r1]: x <- b
[r2]: y <- a
[r3]: z <- a
PREF: a,b > c
''')
print('a>c', p.strictly_preferred('a','c'), '| b>c', p.strictly_preferred('b','c'), '| a>b', p.strictly_preferred('a','b'))
A = p.arguments()
for a in A: print(a)
for t in p.attacks(A): print(t)

a>c True | b>c True | a>b False
A1: {a} ⊢ a
A2: {b} ⊢ b
A3: {c} ⊢ c
A4: {b} ⊢ x
A5: {a} ⊢ y
A6: {a} ⊢ z
A4 -> A1  (normal, via a)
A4 -> A5  (normal, via a)
A4 -> A6  (normal, via a)
A5 -> A2  (normal, via b)
A5 -> A4  (normal, via b)
A6 -> A3  (normal, via c)


## 6. Vérification : l'ABA atomique produit les mêmes arguments

In [9]:
pairs = lambda f: {(a.claim, a.support) for a in f.arguments()}
print(pairs(fw) == pairs(fw.prepare()), pairs(circ) == pairs(circ.prepare()))

True True


## 7. Graphe (Graphviz)

In [10]:
print(fw.to_dot(args))

digraph ABA {
  rankdir=LR;
  node [shape=box, style=rounded, fontname="Helvetica"];
  A1 [label="A1\n{} ⊢ q"];
  A2 [label="A2\n{a} ⊢ a"];
  A3 [label="A3\n{b} ⊢ b"];
  A4 [label="A4\n{c} ⊢ c"];
  A5 [label="A5\n{a} ⊢ p"];
  A6 [label="A6\n{b,c} ⊢ r"];
  A7 [label="A7\n{a,c} ⊢ s"];
  A8 [label="A8\n{a,c} ⊢ t"];
  A2 -> A6 [color="red", style=dashed];
  A5 -> A6 [color="red", style=dashed];
  A7 -> A3 [color="black"];
  A7 -> A6 [color="black"];
  A7 -> A6 [color="red", style=dashed];
  A8 -> A4 [color="black"];
  A8 -> A6 [color="black"];
  A8 -> A6 [color="red", style=dashed];
  A8 -> A7 [color="black"];
  A8 -> A8 [color="black"];
}
